# 📊 Gold Price Prediction — EDA & ML Starter Notebook

**Goal:** Explore the merged Yahoo Finance dataset, build features, and train a baseline machine-learning model to predict next-day gold prices.

---

### Before you run this notebook:
1. Install dependencies: `pip install -r requirements.txt`
2. Download data: `python scripts/download_market_data.py`
3. Start Jupyter: `jupyter notebook`

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
print('Imports OK')

## 1. Load Data

In [ ]:
DATA_PATH = '../data/yahoo_market_data.csv'

df = pd.read_csv(DATA_PATH, index_col='Date', parse_dates=True)

print(f'Shape : {df.shape}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
print(f'\nColumns ({len(df.columns)}):')
print(df.columns.tolist())
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 2a. Basic statistics ──────────────────────────────────────────────────
df.describe()

In [ ]:
# ── 2b. Missing values ────────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — dataset is complete ✓')

In [ ]:
# ── 2c. Gold price over time ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
df['gold_Close'].plot(ax=ax, color='goldenrod', linewidth=1.2)
ax.set_title('Gold Futures — Closing Price', fontsize=14)
ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2d. Multi-asset comparison (normalised to 100 at start) ───────────────
close_cols = [c for c in df.columns if c.endswith('_Close')]
normalised = df[close_cols].div(df[close_cols].iloc[0]).mul(100)

fig, ax = plt.subplots(figsize=(14, 5))
normalised.plot(ax=ax, linewidth=1)
ax.set_title('Normalised Closing Prices (base = 100)', fontsize=14)
ax.set_ylabel('Index')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2e. Correlation heatmap ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[close_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Correlation — Closing Prices', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
feat = df.copy()

# ── Daily log returns ─────────────────────────────────────────────────────
for col in close_cols:
    prefix = col.replace('_Close', '')
    feat[f'{prefix}_ret'] = np.log(feat[col] / feat[col].shift(1))

# ── Rolling averages for gold ─────────────────────────────────────────────
for window in [5, 20, 50]:
    feat[f'gold_ma{window}'] = feat['gold_Close'].rolling(window).mean()
    feat[f'gold_vol{window}'] = feat['gold_ret'].rolling(window).std()  # volatility

# ── Simple RSI (14-day) for gold ──────────────────────────────────────────
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, float('inf'))
    return 100 - (100 / (1 + rs))

feat['gold_rsi14'] = compute_rsi(feat['gold_Close'])

# ── Lag features (gold return, 1–5 days ago) ──────────────────────────────
for lag in range(1, 6):
    feat[f'gold_ret_lag{lag}'] = feat['gold_ret'].shift(lag)

# ── Cross-asset lag returns ───────────────────────────────────────────────
for asset in ['sp500', 'dxy', 'oil', 'tnx']:
    feat[f'{asset}_ret_lag1'] = feat[f'{asset}_ret'].shift(1)

print(f'Feature set shape: {feat.shape}')
feat.tail(3)

## 4. Target Variable Creation

In [ ]:
# ── Option A: Next-day log return (regression) ────────────────────────────
feat['target_ret'] = feat['gold_ret'].shift(-1)   # shift(-1) = next day's return

# ── Option B: Next-day direction (classification) ─────────────────────────
feat['target_dir'] = (feat['target_ret'] > 0).astype(int)  # 1 = up, 0 = down

print('Target distribution (direction):')
print(feat['target_dir'].value_counts())

# Plot return distribution
fig, ax = plt.subplots(figsize=(8, 3))
feat['target_ret'].dropna().hist(bins=80, ax=ax, color='goldenrod', edgecolor='white')
ax.set_title('Distribution of Next-Day Gold Returns')
ax.set_xlabel('Log Return')
plt.tight_layout()
plt.show()

## 5. Prepare ML Dataset

In [ ]:
FEATURE_COLS = [
    # gold technicals
    'gold_ret', 'gold_ma5', 'gold_ma20', 'gold_ma50',
    'gold_vol5', 'gold_vol20', 'gold_rsi14',
    # gold lag returns
    'gold_ret_lag1', 'gold_ret_lag2', 'gold_ret_lag3',
    'gold_ret_lag4', 'gold_ret_lag5',
    # cross-asset lag returns
    'sp500_ret_lag1', 'dxy_ret_lag1', 'oil_ret_lag1', 'tnx_ret_lag1',
]

TARGET_COL = 'target_ret'   # swap to 'target_dir' for classification

ml = feat[FEATURE_COLS + [TARGET_COL]].dropna()
print(f'ML dataset: {ml.shape[0]:,} rows × {ml.shape[1]} columns')

X = ml[FEATURE_COLS]
y = ml[TARGET_COL]

# ── Time-series split (no shuffling!) ─────────────────────────────────────
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

## 6. Baseline Model — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_r2   = r2_score(y_test, lr_pred)

print('=== Linear Regression ===')
print(f'  RMSE : {lr_rmse:.6f}')
print(f'  MAE  : {lr_mae:.6f}')
print(f'  R²   : {lr_r2:.4f}')

## 7. Random Forest Model

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)

print('=== Random Forest ===')
print(f'  RMSE : {rf_rmse:.6f}')
print(f'  MAE  : {rf_mae:.6f}')
print(f'  R²   : {rf_r2:.4f}')

## 8. Model Comparison & Feature Importance

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'RMSE': [lr_rmse, rf_rmse],
    'MAE':  [lr_mae,  rf_mae],
    'R²':   [lr_r2,   rf_r2],
})
results

In [ ]:
# ── Feature importances (Random Forest) ──────────────────────────────────
importance = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
importance.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Feature Importances — Random Forest')
ax.set_ylabel('Importance')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(importance.to_string())

In [ ]:
# ── Actual vs Predicted plot (Random Forest) ──────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
pd.Series(y_test.values, index=y_test.index, name='Actual').plot(ax=ax, label='Actual', alpha=0.7)
pd.Series(rf_pred, index=y_test.index, name='Predicted').plot(ax=ax, label='RF Predicted', alpha=0.7)
ax.set_title('Actual vs Predicted — Next-Day Gold Return (Test Set)')
ax.set_ylabel('Log Return')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Next Steps

Now that you have a working baseline, here are ideas for improving the model:

| Idea | How |
|------|-----|
| Add more technical indicators | Use the `ta` library: `ta.momentum.RSIIndicator`, `ta.trend.MACD`, etc. |
| Tune Random Forest | Use `GridSearchCV` or `RandomizedSearchCV` |
| Try Gradient Boosting | `from sklearn.ensemble import GradientBoostingRegressor` or install XGBoost/LightGBM |
| Time-series cross-validation | Replace the single split with `TimeSeriesSplit` |
| Try a classification target | Change `TARGET_COL = 'target_dir'` and use a classifier |
| Add macro data | Download CPI, PMI or other economic indicators |

Good luck with your ML project! 🥇